In [71]:
# Checking Python executable
import sys
print(sys.executable) # MAKE SURE THIS POINTS TO THE CORRECT VIRTUAL ENVIRONMENT PATH FOR CORRECT PACKAGE INSTALLATION

/home/louis/miniconda3/envs/aml_lab/bin/python


In [72]:
# ALWAYS INSTALL USING %pip, NOT !pip (can sometimes install to system Python) or pip
# %pip install numpy
# %pip install pandas
# %pip install matplotlib
# %pip install scikit-learn
# %pip install torch # Using version 2.10.0+cu128
# %pip install torchinfo

In [73]:
# Import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchinfo
print(torch.__version__)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Define device (either GPU or CPU if GPU is unavailable)
# DEVICE = "cpu"

##### Model training function #####
def train(
        model: nn.Module,
        train_loader: DataLoader,
        criterion: nn.Module,
        optimizer: torch.optim.Optimizer,
        num_epochs: int = 10,
        val_loader: DataLoader = None,
        device: torch.device = DEVICE,
        print_loss: bool = True, # Flag for whether to print loss outputs or not
):
    model = model.to(device) # Move the model to same device as data (GPU or CPU)

    # Train for the number of epochs specified
    for epoch in range(num_epochs):

        ### TRAINING SET ###
        model.train() # Set model to training mode (affects Dropout/BatchNorm)
        train_loss = 0.0 # Initialise training loss

        # Loop through all batches in the training set
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device) # Move data to same device as model (GPU or CPU)
            optimizer.zero_grad() # Clear gradients
            preds = model(inputs) # Forward pass, obtain predictions
            loss = criterion(preds, labels) # Compute loss based on predictions and true labels (loss is MEAN loss over the batch)
            loss.backward() # Backward pass, compute gradient of loss w.r.t every model parameter
            optimizer.step() # Update weights, using optimisation algorithm chosen

            train_loss += loss.item() * inputs.size(0) # Sum training loss of EACH SAMPLE in the batch (inputs.size(0) is batch size)
        train_loss /= len(train_loader.dataset) # Calculate mean loss PER SAMPLE over ENTIRE DATASET

        ### VALIDATION SET ###
        # Loop through all validation batches (if validation data is given)
        if val_loader is not None:
            model.eval() # Set model to evaluation (inference) mode (turns dropout OFF, and affects BatchNorm)
            val_loss = 0.0 # Initialise validation loss
            all_preds_class = [] # Initialise list to store output prediction classes
            all_labels = [] # Initialise list to store actual labels of output predictions

            with torch.no_grad(): # Disable gradient computing
                # Loop through all batches in the validation set
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device) # Move data to same device as model (GPU or CPU)
                    preds = model(inputs) # Forward pass, obtain predictions as LOGITS (NO FOLLOWING BACKWARD PASS IN VALIDATION)

                    # Compute confusion matrix values
                    preds_class = torch.argmax(preds, dim=1) # Get class index of logit predictions
                    all_preds_class.append(preds_class.cpu())
                    all_labels.append(labels.cpu())

                    # Compute validation loss
                    loss = criterion(preds, labels) # Compute loss based on predictions and true labels (loss is MEAN loss over the batch)
                    val_loss += loss.item() * inputs.size(0) # Sum validation loss of EACH SAMPLE in the batch (inputs.size(0) is batch size)
                val_loss /= len(val_loader.dataset)

                all_preds_class = torch.cat(all_preds_class)
                all_labels = torch.cat(all_labels)

                cm = confusion_matrix(all_labels, all_preds_class)
                print("Validation confusion matrix:\n", cm)

        ### PRINT TRAINING/VALIDATION OUTPUTS ###
            if print_loss:
                print(f"Epoch[{epoch+1}/{num_epochs}] Training Loss: {train_loss:.5f}, Validation Loss: {val_loss:.5f}")
        else:
            if print_loss:
                print(f"Epoch[{epoch+1}/{num_epochs}] Training Loss: {train_loss:.5f}")


##### Model evaluation function #####
def eval(
        model: nn.Module,
        test_loader: DataLoader,
        criterion: nn.Module,
        device: torch.device = DEVICE,
        print_loss: bool = True, # Flag for whether to print loss outputs or not
):
    model.eval() # Set model to evaluation mode
    test_loss = 0.0 # Initialise test loss
    all_preds_class = [] # Initialise list to store output prediction classes
    all_labels = [] # Initialise list to store actual labels of output predictions

    with torch.no_grad(): # Disable gradient computing
        # Loop through all batches in the test set
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device) # Move data to same device as model (GPU or CPU)
            preds = model(inputs) # Forward pass, obtain predictions as LOGITS (NO FOLLOWING BACKWARD PASS IN TESTING)

            # Compute confusion matrix values
            preds_class = torch.argmax(preds, dim=1) # Get class index of logit predictions
            all_preds_class.append(preds_class.cpu())
            all_labels.append(labels.cpu())
            
            # Compute validation loss
            loss = criterion(preds, labels) # Compute loss based on predictions and true labels (loss is MEAN loss over the batch)

            test_loss += loss.item() * inputs.size(0) # Sum test loss of EACH SAMPLE in the batch (inputs.size(0) is batch size)
        test_loss /= len(test_loader.dataset)

        all_preds_class = torch.cat(all_preds_class)
        all_labels = torch.cat(all_labels)

        cm = confusion_matrix(all_labels, all_preds_class)
        print("Testing confusion matrix:\n", cm)

    if print_loss:
        print(f"Test Loss: {test_loss:.5f}")

2.10.0+cu128


In [74]:
### Import data ###
# Remove first two columns (they are just host_time and time)
# data1 = pd.read_csv("Data/adi_focused.csv")
# data2 = pd.read_csv("Data/adi_stressed.csv")
# data3 = pd.read_csv("Data/louis_focused.csv")
# data4 = pd.read_csv("Data/louis_stressed.csv")
base_data1_raw = pd.read_csv("Data/adi_7_5min_baseline.csv").iloc[:, 2:]
base_data2_raw = pd.read_csv("Data/emmanuel_7_5min_baseline.csv").iloc[:, 2:]
base_data3_raw = pd.read_csv("Data/louis_7_5min_baseline.csv").iloc[:, 2:]

dist_data1_raw = pd.read_csv("Data/adi_7_5min_distraction.csv").iloc[:, 2:]
dist_data2_raw = pd.read_csv("Data/emmanuel_7_5min_diistraction.csv").iloc[:, 2:]
dist_data3_raw = pd.read_csv("Data/louis_7_5min_distract.csv").iloc[:, 2:]

foc_data1_raw = pd.read_csv("Data/adi_7_5min_focus.csv").iloc[:, 2:]
foc_data2_raw = pd.read_csv("Data/emmanuel_7_5min_focus.csv").iloc[:, 2:]
foc_data3_raw = pd.read_csv("Data/louis_7_5min_focus.csv").iloc[:, 2:]

str_data1_raw = pd.read_csv("Data/adi_7_5min_stress.csv").iloc[:, 2:]
str_data2_raw = pd.read_csv("Data/emmanuel_7_5min_stress.csv").iloc[:, 2:]
str_data3_raw = pd.read_csv("Data/louis_7_5min_stress.csv").iloc[:, 2:]

# DEBUGGING
# base_data1.head()

In [75]:
##### Process data #####
### Function for normalising data ###
def normalise_data(data_in): # INPUT DATA IS A PANDAS DATAFRAME
    return (data_in - data_in.mean()) / data_in.std()

### Function for obtaining training, validation and test splits
def split_data(data_in, train_split, val_split, test_split): # data_in is a list of pandas dataframes
    # Create empty pandas dataframes for storing training, validation and test data splits
    train_data = pd.DataFrame()
    val_data = pd.DataFrame()
    test_data = pd.DataFrame()

    # Loop through all datasets in the input list of datasets
    for dataset in data_in:
        # Obtain number of training, validation and test data
        train_num = round(dataset.shape[0]*train_split)
        val_num = round(dataset.shape[0]*val_split)
        test_num = dataset.shape[0] - train_num - val_num

        # Obtain the training, validation and test data splits
        train_data_curr = dataset.iloc[0:train_num]
        val_data_curr = dataset.iloc[train_num:train_num+val_num]
        test_data_curr = dataset.iloc[train_num+val_num:]

        # Concatenate current dataset's training, validation and test data to the overall data
        train_data = pd.concat([train_data, train_data_curr], axis=0, ignore_index=True)
        val_data = pd.concat([val_data, val_data_curr], axis=0, ignore_index=True)
        test_data = pd.concat([test_data, test_data_curr], axis=0, ignore_index=True)
    
    # Convert pandas dataframes to numpy arrays
    train_data_np = train_data.values.astype("float32")
    val_data_np = val_data.values.astype("float32")
    test_data_np = test_data.values.astype("float32")

    return train_data_np, val_data_np, test_data_np

### Function for obtaining array of class labels for a dataset
def create_labels(data_in, label):
    all_labels = [] # Initialise

    # Loop through all training, validation and test datasets
    for dataset in data_in:
        dataset_labels = np.ones(len(dataset), dtype=np.int64)*label # Create an array of desired labels for the current dataset
        all_labels.append(dataset_labels) # Append labels to output

    return all_labels[0], all_labels[1], all_labels[2] # Return array of labels for training, validation and test sets

### Function for obtaining windowed input-label pairs FOR A SINGLE DATASET (needed as our data is highly dependent on previous data) ###
# E.g. [x0, x1, x2] -> y       [x1, x2, x3] -> y ...
def window_single_data(dataset, labels, win_size=10):
    # print(range(len(dataset) - win_size))
    # print(dataset)
    input_seq = np.array([dataset[i:i+win_size, :] for i in range(len(dataset) - win_size)]) # Get input sequence with length = window length
    seq_label = [labels[i+win_size, 0] for i in range(len(dataset) - win_size)] # Get corresponding output for each input window sequence

    # print(input_seq)
    return np.array(input_seq), np.array(seq_label)

### Function for obtaining windowed input-label pairs FOR A LIST OF DATASETS ###
def window_data(data_in, labels_in, win_size=10):
    # Initialisations
    all_data = []
    all_labels = []

    # Loop through all datasets (in the data_in list)
    for dataset, labels in zip(data_in, labels_in):
        windowed_dataset, windowed_labels = window_single_data(dataset, np.vstack(labels), win_size) # Get windowed input-label pairs for current dataset
        all_data.append(windowed_dataset) # Store current windowed dataset
        all_labels.append(windowed_labels) # Store current windowed labels
    
    data_out = np.concatenate(all_data, axis=0) # Concatenate all windowed dataset
    labels_out = np.concatenate(all_labels, axis=0) # Concatenate all windowed labels

    return data_out, labels_out


# Initialisations
num_sensor_readings = 20 # Number of sensor readings
num_classes = 4 # Number of classification classes (number of emotional states to identify)
window_size = 16

# Normalise all data
base_data1 = normalise_data(base_data1_raw)
base_data2 = normalise_data(base_data2_raw)
base_data3 = normalise_data(base_data3_raw)

dist_data1 = normalise_data(dist_data1_raw)
dist_data2 = normalise_data(dist_data2_raw)
dist_data3 = normalise_data(dist_data3_raw)

foc_data1 = normalise_data(foc_data1_raw)
foc_data2 = normalise_data(foc_data2_raw)
foc_data3 = normalise_data(foc_data3_raw)

str_data1 = normalise_data(str_data1_raw)
str_data2 = normalise_data(str_data2_raw)
str_data3 = normalise_data(str_data3_raw)

# Group all data
base_data = [base_data1, base_data2, base_data3]
dist_data = [dist_data1, dist_data2, dist_data3]
foc_data = [foc_data1, foc_data2, foc_data3]
str_data = [str_data1, str_data2, str_data3]

# Obtain training, validation and test splits
base_train_data, base_val_data, base_test_data = split_data(base_data, 0.8, 0.1, 0.1)
dist_train_data, dist_val_data, dist_test_data = split_data(dist_data, 0.8, 0.1, 0.1)
foc_train_data, foc_val_data, foc_test_data = split_data(foc_data, 0.8, 0.1, 0.1)
str_train_data, str_val_data, str_test_data = split_data(str_data, 0.8, 0.1, 0.1)
# print(base_train_data)

# Obtain labels for training, validation and test splits
base_train_labels, base_val_labels, base_test_labels = create_labels([base_train_data, base_val_data, base_test_data], 0)
dist_train_labels, dist_val_labels, dist_test_labels = create_labels([dist_train_data, dist_val_data, dist_test_data], 1)
foc_train_labels, foc_val_labels, foc_test_labels = create_labels([foc_train_data, foc_val_data, foc_test_data], 2)
str_train_labels, str_val_labels, str_test_labels = create_labels([str_train_data, str_val_data, str_test_data], 3)
# print(str_test_labels)

# Place all training, validation and testing data and labels into lists
train_data_list = [base_train_data, dist_train_data, foc_train_data, str_train_data]
train_labels_list = [base_train_labels, dist_train_labels, foc_train_labels, str_train_labels]

val_data_list = [base_val_data, dist_val_data, foc_val_data, str_val_data]
val_labels_list = [base_val_labels, dist_val_labels, foc_val_labels, str_val_labels]

test_data_list = [base_test_data, dist_test_data, foc_test_data, str_test_data]
test_labels_list = [base_test_labels, dist_test_labels, foc_test_labels, str_test_labels]

# Window the training, validation and test splits
train_data, train_labels = window_data(train_data_list, train_labels_list, window_size)
val_data, val_labels = window_data(val_data_list, val_labels_list, window_size)
test_data, test_labels = window_data(test_data_list, test_labels_list, window_size)
print(train_data.shape)


# ### DEBUGGING
# # print(train_data_raw.head()) # Print first 5 rows of data for inspection
# # print(test_data_raw.head()) # Print first 5 rows of data for inspection
# # print(test_data.shape)
# # plt.plot(train_data_np[:,2])

(20939, 16, 20)


In [76]:
##### Create dataloaders #####
# Convert data to tensors
train_inputs_tensor = torch.tensor(train_data, dtype=torch.float32)
train_labels_tensor = torch.tensor(train_labels, dtype=torch.long)#.unsqueeze(1) # nn.CrossEntropyLoss() expects integer class labels, no floats or one-hot. Also no need for unsqueeze(1) for CrossEntropyLoss, it just take labels with dim [batch size]

val_inputs_tensor = torch.tensor(val_data, dtype=torch.float32)
val_labels_tensor = torch.tensor(val_labels, dtype=torch.long)#.unsqueeze(1)

test_inputs_tensor = torch.tensor(test_data, dtype=torch.float32)
test_labels_tensor = torch.tensor(test_labels, dtype=torch.long)#.unsqueeze(1)

# Build data loaders
train_dataset = TensorDataset(train_inputs_tensor, train_labels_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_dataset = TensorDataset(val_inputs_tensor, val_labels_tensor)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)#True)

test_dataset = TensorDataset(test_inputs_tensor, test_labels_tensor)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)#True)


## DEBUGGING
print(f"Shape of training inputs: {train_inputs_tensor.shape}")
print(f"Shape of training labels: {train_labels_tensor.shape}")

print(f"Shape of validation inputs: {val_inputs_tensor.shape}")
print(f"Shape of validation labels: {val_labels_tensor.shape}")

print(f"Shape of testing inputs: {test_inputs_tensor.shape}")
print(f"Shape of testing labels: {test_labels_tensor.shape}")

Shape of training inputs: torch.Size([20939, 16, 20])
Shape of training labels: torch.Size([20939])
Shape of validation inputs: torch.Size([2560, 16, 20])
Shape of validation labels: torch.Size([2560])
Shape of testing inputs: torch.Size([2561, 16, 20])
Shape of testing labels: torch.Size([2561])


In [77]:
print(f"Train labels min: {train_labels_tensor.min()}, max: {train_labels_tensor.max()}")
print(f"Train labels shape: {train_labels_tensor.shape}")

Train labels min: 0, max: 3
Train labels shape: torch.Size([20939])


In [78]:
##### Model definition #####
class LSTMClassifier(nn.Module):
    def __init__(self, input_size=num_sensor_readings, hidden_size=64, output_size=num_classes): # Note: output_size should be equal to number of classification classes
        super().__init__()
        self.lstm1 = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True, dropout=0.3) # Input: [batch, sequence length, input dimension (number of sensors)]
        # self.relu1 = nn.ReLU() # ReLU
        self.dropout1 = nn.Dropout(0.2) # Dropout

        self.lstm2 = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size, batch_first=True, dropout=0.3) # Input: [batch, sequence length, input dimension (number of sensors)]
        # self.relu2 = nn.ReLU() # ReLU
        self.dropout2 = nn.Dropout(0.2) # Dropout

        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.layernorm1 = nn.LayerNorm(hidden_size) # Layer norm
        self.relu3 = nn.ReLU() # ReLU
        self.dropout3 = nn.Dropout(0.2) # Dropout

        self.fc2 = nn.Linear(hidden_size, output_size)
        # self.layernorm2 = nn.LayerNorm(output_size) # Layer norm
        # self.relu4 = nn.ReLU() # ReLU
        # self.dropout4 = nn.Dropout(0.3) # Dropout
    
    def forward(self, x):
        out, _ = self.lstm1(x) # Output: [batch, sequence length, hidden dimension (number of sensors)]
        # out = out[:, -1, :] # Use final hidden state of model as the output classification (Output: [batch, hidden dimension])
        # out = self.relu1(out)
        out = self.dropout1(out)
        
        out, _ = self.lstm2(out)
        # out = self.relu2(out)
        out = self.dropout2(out)
        out = out[:, -1, :]
        
        out = self.fc1(out) # CLASSIFY: Output here are LOGITS (Output: [batch, num_classes])
        out = self.layernorm1(out)
        out = self.relu3(out)
        out = self.dropout3(out)

        out = self.fc2(out) # CLASSIFY: Output here are LOGITS (Output: [batch, num_classes])
        # out = self.layernorm2(out)
        # out = self.relu4(out)
        # out_logits = self.dropout4(out)

        return out # AS LOGITS

In [79]:
##### Traing model #####
model = LSTMClassifier()#.to(DEVICE) # Define model
print(torchinfo.summary(model, input_size=(1, window_size, num_sensor_readings))) # Input: [batch size, sequence length, input size (number of sensors)]

# Train model
train(
    model,
    train_loader,
    nn.CrossEntropyLoss(), #nn.CrossEntropyLoss() for multi-class classification, nn.BCEWithLogitsLoss() for binary classification, nn.MSELoss()
    optim.Adam(model.parameters(), lr=0.001),
    num_epochs=50,#500
    val_loader=val_loader,
    print_loss=True,
)


# Test model
eval(
    model,
    test_loader,
    nn.CrossEntropyLoss(), #nn.CrossEntropyLoss() for multi-class classification, nn.BCEWithLogitsLoss() for binary classification, nn.MSELoss()
    print_loss=True,
)

Layer (type:depth-idx)                   Output Shape              Param #
LSTMClassifier                           [1, 4]                    --
├─LSTM: 1-1                              [1, 16, 64]               22,016
├─Dropout: 1-2                           [1, 16, 64]               --
├─LSTM: 1-3                              [1, 16, 64]               33,280
├─Dropout: 1-4                           [1, 16, 64]               --
├─Linear: 1-5                            [1, 64]                   4,160
├─LayerNorm: 1-6                         [1, 64]                   128
├─ReLU: 1-7                              [1, 64]                   --
├─Dropout: 1-8                           [1, 64]                   --
├─Linear: 1-9                            [1, 4]                    260
Total params: 59,844
Trainable params: 59,844
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.89
Input size (MB): 0.00
Forward/backward pass size (MB): 0.02
Params size (MB): 0.24
Estimated Total Siz

/home/louis/miniconda3/envs/aml_lab/lib/python3.14/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.3 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Validation confusion matrix:
 [[290  59  13 135]
 [ 76 502  20 105]
 [117  64 417  93]
 [107  39 112 411]]
Epoch[1/50] Training Loss: 0.54156, Validation Loss: 1.55636
Validation confusion matrix:
 [[304  73  17 103]
 [ 23 551   5 124]
 [ 66 106 404 115]
 [ 89  31 124 425]]
Epoch[2/50] Training Loss: 0.08205, Validation Loss: 1.99465
Validation confusion matrix:
 [[380   5   3 109]
 [ 23 551   5 124]
 [172  10 373 136]
 [ 71  99  62 437]]
Epoch[3/50] Training Loss: 0.03279, Validation Loss: 2.06471
Validation confusion matrix:
 [[342  65   1  89]
 [ 43 557   5  98]
 [174  53 341 123]
 [119  90  71 389]]
Epoch[4/50] Training Loss: 0.02748, Validation Loss: 2.35895
Validation confusion matrix:
 [[379  15   0 103]
 [ 39 558  10  96]
 [152  18 337 184]
 [154  24  57 434]]
Epoch[5/50] Training Loss: 0.01866, Validation Loss: 2.32637
Validation confusion matrix:
 [[314  70   0 113]
 [ 30 583   4  86]
 [146  43 369 133]
 [ 95 126  47 401]]
Epoch[6/50] Training Loss: 0.02403, Validation Loss: 